# Repository Walkthrough

Questo notebook documenta la struttura e il funzionamento di `LLM-Needs-a-Plan` nella copia presente in questa repo.

Obiettivi:
- spiegare la gerarchia delle cartelle
- chiarire a cosa serve ogni sottoparte
- descrivere la pipeline end-to-end
- aggiungere una lettura piu fine dei file principali

Scelta del posizionamento:
- il notebook e stato messo nella root del sottoprogetto, accanto a `Results Analysis.ipynb`
- qui e facile da trovare insieme agli altri materiali di documentazione e analisi
- non e dentro `src/` perche non fa parte del codice runtime
- non e dentro `assets/` perche non e una risorsa statica, ma un documento di studio della repo


## 1. Panorama della repo principale

La repo radice `LLM_Benchmark` e un workspace di progetto, non una singola applicazione. Le sue tre aree principali sono:

- `References/`: contiene repo esterne o snapshot di lavori di riferimento. Qui si trova `LLM-Needs-a-Plan-main`, che e il sottoprogetto operativo piu importante per il benchmark.
- `Planning Domains/`: raccoglie domini PDDL aggiuntivi, utili per estendere il benchmark oltre i due domini gia integrati.
- `Papers/`: contiene articoli e riferimenti bibliografici sul planning con LLM.

Quindi il progetto vero e proprio, per come e organizzato questo workspace, sta in:

```text
References/LLM-Needs-a-Plan-main/LLM-Needs-a-Plan-main/
```


## 2. Gerarchia di `LLM-Needs-a-Plan`

La struttura osservata nella copia locale e questa:

```text
LLM-Needs-a-Plan-main/
|-- README.md
|-- config.yml
|-- requirements.txt
|-- requirements_analysis.txt
|-- report.pdf
|-- Results Analysis.ipynb
|-- Repository Walkthrough.ipynb
|-- assets/
|   |-- Guida Leonardo.pdf
|   |-- images/
|   |-- literature/
|   `-- tutorials/
|-- scripts/
|   |-- gemma3_*.sh
|   |-- kimi_*.sh
|   |-- llama3_*.sh
|   `-- phi4_*.sh
`-- src/
    |-- main.py
    |-- core/
    |-- data/
    |-- prompts/
    |-- results/
    |-- tests/
    `-- utils/
```

Ruolo dei blocchi principali:
- file in root: configurazione, dipendenze, report e notebook di analisi
- `assets/`: materiale di supporto e onboarding sul cluster Leonardo
- `scripts/`: launcher Bash/SLURM per eseguire gli esperimenti
- `src/`: codice, dati, prompt, risultati e test

Nota importante sulla copia attuale:
- README, config e test fanno riferimento anche a `src/models/` e `VAL/`
- nella copia locale analizzata queste due directory non sono presenti
- quindi questa snapshot contiene bene codice, dati, prompt e risultati, ma non tutti gli asset necessari per rieseguire gli esperimenti end-to-end senza integrazioni esterne


## 3. Come funziona la repo in generale

La pipeline completa e pensata per risolvere problemi di planning PDDL con LLM locali, validare il piano prodotto e poi analizzare i risultati.

Flusso end-to-end:
1. `src/main.py` legge `config.yml` e gli argomenti CLI.
2. `PDDLPlanner` scopre i domini e le istanze in `src/data/`.
3. `ModelManager` carica un modello locale da `weights_path`.
4. `PDDLProcessor` costruisce il prompt corretto per il dominio scelto.
5. Il modello genera un piano in linguaggio PDDL.
6. `answer_postprocessor.py` prova a estrarre solo le azioni valide dal testo del modello.
7. `validator.py` invoca VAL per controllare se il piano e valido.
8. Se il piano non valida, il sistema costruisce un feedback e riprova per piu iterazioni.
9. Il piano finale viene salvato in `src/results/<model>/<domain>/instance-xx_plan.txt` con metadati.
10. `Results Analysis.ipynb` legge questi file e produce statistiche e grafici.

In altre parole, la repo ha tre livelli logici:
- livello dati: file PDDL in `src/data/`
- livello esecuzione: codice in `src/core/`, `src/utils/`, `src/prompts/`
- livello analisi: risultati in `src/results/` e notebook/immagini in root e `assets/images/`

Configurazione centrale in `config.yml`:
- `MODEL_PATH`: cartella dei pesi del modello
- `PROBLEMS_PATH`: cartella dei domini PDDL
- `MODEL_OUTPUT`: cartella base dei risultati
- `VAL_PATH`, `VAL_EXECUTABLE`, `VAL_TIMEOUT`: configurazione del validator VAL
- `MAX_TOKENS`, `TEMPERATURE`, `TOP_K`: parametri di generazione
- `DEFAULT_ITERATIONS`: numero massimo di tentativi di correzione


## 4. Lettura piu fine: entrypoint e moduli `core`

### `src/main.py`
- e il punto di ingresso CLI del framework
- costruisce il parser degli argomenti usando come default i valori di `config.yml`
- verifica l'esistenza di `problems_path` e `weights_path`
- risolve `output_dir`, aggiungendo l'alias del modello alla cartella di output
- inizializza `PDDLPlanner`, poi chiama `setup()` e `run()`

### `src/core/file_manager.py`
- definisce `DomainBundle`, una dataclass immutabile che rappresenta un dominio con il suo file PDDL e le sue istanze
- `read_file()` legge un file testuale con gestione errori
- `save_file()` salva contenuti su disco creando le directory necessarie
- `find_pddl_files()` e il punto chiave: scopre i domini sotto `src/data/`
- `_locate_domain_file()` riconosce sia `domain.pddl` sia file del tipo `<dominio>_domain.pddl`
- `_collect_problem_files()` raccoglie tutti i `.pddl` che non sono il file di dominio

Dettaglio utile:
- il file manager sa gestire sia il caso `src/data/` con piu domini, sia il caso in cui il path passato punti gia a un singolo dominio

### `src/core/model_manager.py`
- rileva il tipo di modello dal nome della cartella pesi: `phi4`, `llama3`, `gemma3`, `kimi`
- `load()` carica modello e tokenizer Hugging Face
- se c'e CUDA usa `device_map='auto'`, altrimenti fa fallback su CPU
- `generate_response()` applica il chat template del tokenizer, oppure un fallback minimale se il tokenizer non lo supporta
- `_build_generation_config()` separa chiaramente generazione greedy e sampling
- `iterative_planning_with_validation()` implementa il loop piu importante della repo

Nel loop iterativo:
- costruisce i messaggi iniziali con eventuale system prompt
- genera una risposta
- usa `formatter()` per estrarre il piano
- valida il piano con VAL
- se il piano e invalido, aggiunge feedback nel contesto e riprova
- prima di riaggiungere la risposta del modello, la ripulisce per rimuovere eventuali blocchi `<think>` troppo lunghi

Dettaglio molto concreto:
- anche se l'utente passa `include_prompt=True`, nel loop iterativo il codice forza `include_prompt=False` per evitare che la risposta contenga il prompt e faccia crescere troppo il contesto

### `src/core/pddl_processor.py`
- costruisce i prompt, gestisce il dominio corrente e salva i risultati
- `process_domain_with_validation()` itera su tutte le istanze di un dominio
- `_process_single_problem()` legge il problema, costruisce il prompt, invoca il modello, riceve il piano e lo salva
- `_build_prompt()` compone istruzioni, esempi few-shot, vincoli opzionali e Chain-of-Thought opzionale
- `_create_domain_prompt()` sceglie fra prompt specializzati per `tetris`, `citycar` o prompt generico
- `_get_validation_feedback_fn()` sceglie il messaggio di correzione domain-specifico da dare al modello dopo un errore di validazione

Dettagli di comportamento:
- il dominio viene riconosciuto con un controllo semplice sul nome della cartella, quindi basta che il nome contenga `tetris` o `citycar`
- i piani vengono salvati con una sezione finale standard `--- Processing Metadata ---`, poi usata dal notebook di analisi

### `src/core/pddl_planner.py`
- e l'orchestratore di alto livello
- `setup()` crea `FileManager`, scopre i domini, applica eventuale filtro per nome dominio, carica il modello e crea `PDDLProcessor`
- `run()` esegue o la modalita batch o la modalita sequenziale
- `_build_processing_kwargs()` centralizza i parametri che vengono passati al processor
- `_log_summary()` costruisce il riepilogo finale per problemi totali, successi e fallimenti

In pratica:
- `main.py` e il front door
- `PDDLPlanner` e il direttore d'orchestra
- `PDDLProcessor` prepara il lavoro per ogni problema
- `ModelManager` produce e corregge il piano
- `FileManager` fornisce input e salva output


## 5. Lettura piu fine: `utils` e `prompts`

### `src/utils/configuration.py`
- cerca `config.yml` nella root del progetto e, in fallback, sotto `src/`
- aggiunge `src` al `sys.path` per rendere piu semplici gli import
- espone `load_config()` come punto unico di caricamento configurazione

### `src/utils/logging_utils.py`
- centralizza la configurazione del logging
- supporta output su stdout e, opzionalmente, su file
- espone `configure_logging()` e `get_logger()`

### `src/utils/common_utils.py`
- contiene helper generici per YAML e filesystem
- `load_yaml_file()` usa `yaml.safe_load`
- `save_yaml_file()` salva dati in YAML
- `ensure_directory_exists()` crea directory ricorsivamente

### `src/utils/answer_postprocessor.py`
- e il modulo che prova a trasformare una risposta LLM in un piano PDDL pulito
- `formatter()` restituisce un dizionario con piano, reasoning, confidence e problemi di formato
- `extract_plan_actions()` prova piu strategie: sezioni chiamate plan, azioni tra parentesi, liste numerate, righe che iniziano con `(`
- `clean_response_text()` rimuove blocchi `<think>`, blocchi markdown e altre scorie testuali
- `detect_format_issues()` segnala problemi come parentesi sbilanciate o risposte troppo lunghe

Questo modulo e importante perche i modelli non sempre rispondono in un formato perfettamente eseguibile.

### `src/utils/validator.py`
- risolve il path dell'eseguibile VAL partendo da `config.yml`
- se il binario non esiste nel path atteso, prova il fallback al system path
- `validate_plan()` lancia VAL come processo esterno
- `validate_plan_from_text()` crea un file temporaneo `.plan`, poi chiama `validate_plan()`

Questa scelta e pratica: il resto del codice puo lavorare in memoria con stringhe di piano, mentre VAL continua a ricevere un file come input.

### `src/prompts/prompts.py`
- definisce il system prompt di base per il planning PDDL
- contiene prompt specializzati per `tetris` e `citycar`
- aggiunge esempi few-shot per i due domini
- espone prompt di Chain-of-Thought e prompt di correzione dopo errore di validazione
- mantiene anche funzioni generiche di compatibilita come `generic_pddl_prompt()` e `validation_feedback_prompt()`

Dettagli interessanti:
- `tetris` e `citycar` hanno esempi espliciti di istanza + piano atteso
- il system prompt chiede piani corti e a basso costo
- i prompt di feedback non dicono solo 'hai sbagliato', ma ricordano anche i vincoli tipici del dominio


## 6. Dati, risultati, script, test e asset

### `src/data/`
- contiene il benchmark PDDL usato dal framework
- `citycar/` ha 1 file di dominio e 20 istanze
- `tetris/` ha 1 file di dominio e 20 istanze

Lettura concettuale dei domini:
- `citycar`: planning urbano. Le azioni principali riguardano avvio delle auto, costruzione o distruzione di strade, movimenti attraverso junction e arrivo finale.
- `tetris`: planning spaziale. Le azioni muovono pezzi su una griglia e includono anche rotazioni o riconfigurazioni di pezzi composti.

Entrambi i domini usano una metrica di costo (`total-cost`), quindi il problema non e solo arrivare al goal, ma farlo in modo efficiente.

### `src/results/`
- contiene un archivio gia popolato di esperimenti
- la gerarchia e `src/results/<model>/<domain>/<instance>_plan.txt`
- ogni file contiene il piano e una sezione finale di metadati

Snapshot trovato nella repo:
- 160 file piano totali
- 4 modelli: `gemma3`, `kimi`, `llama3`, `phi4`
- 2 domini: `citycar`, `tetris`
- 20 istanze per dominio per ogni modello

Success rate osservato nei file presenti:

| Model | Citycar | Tetris |
| --- | --- | --- |
| `gemma3` | 3/20 validi (15%) | 4/20 validi (20%) |
| `kimi` | 16/20 validi (80%) | 16/20 validi (80%) |
| `llama3` | 16/20 validi (80%) | 17/20 validi (85%) |
| `phi4` | 5/20 validi (25%) | 13/20 validi (65%) |

Esempi di problemi piu facili nella snapshot:
- `citycar/instance-01` ha 100% di successo sui modelli presenti
- `tetris/instance-01`, `instance-11` e `instance-12` hanno 100% di successo

Esempi di problemi piu difficili nella snapshot:
- `citycar/instance-10`, `11`, `13`, `15`, `17`, `19`, `20` hanno 0% di successo
- `tetris/instance-05`, `07`, `15` hanno 0% di successo

### `scripts/`
- contiene script Bash pensati per SLURM su cluster Leonardo
- ogni script seleziona modello, dominio, numero di iterazioni e parametri di generazione
- gli script assumono ambiente Linux, moduli HPC, GPU e virtual environment preesistente

In pratica questi script sono i launcher di produzione, non utility locali per Windows.

### `src/tests/`
- `test_file_manager.py`: controlla discovery dei domini e coerenza della struttura dati
- `test_model_manager.py`: ispeziona directory modello e disponibilita dei file attesi
- `test_pddl_processor.py`: prova l'integrazione con un model manager fittizio
- `test_pddl_planner.py`: smoke test sul planner e sulla struttura degli argomenti
- `test_cluster.py`: test piu orientato a Leonardo e all'ambiente di esecuzione
- `test_val_integration.py`: verifica l'integrazione con VAL

Osservazione: questi file sono utili per capire il progetto, ma molti sono piu vicini a script diagnostici o smoke test che a una suite formale di test unitari completa.

### `assets/`
- `literature/`: paper che motivano il progetto
- `tutorials/`: guida passo passo per usare Leonardo
- `images/`: grafici e immagini gia esportate per il report o la presentazione
- `Guida Leonardo.pdf`: guida esterna al supercomputer


## 7. Osservazioni pratiche e limiti della snapshot

Punti forti dell'organizzazione:
- separazione chiara fra codice, dati, prompt, risultati e materiale di supporto
- pipeline concettualmente semplice da seguire
- salvataggio dei metadati nei file risultato, utile per analisi successive
- presenza di prompt specializzati per dominio, quindi non tutto viene delegato a un prompt generico

Punti da tenere a mente:
- la documentazione descrive anche componenti non presenti in questa copia, in particolare `src/models/` e `VAL/`
- `Results Analysis.ipynb` contiene un `PROJECT_ROOT` hardcoded Linux (`/home/aless/PROJECTS/LLM-Needs-a-Plan`), quindi non e subito portabile cosi com'e
- nello stato salvato del notebook compare anche un errore `ModuleNotFoundError: No module named 'pandas'`
- gli script di esecuzione sono pensati per SLURM e non per un uso locale Windows immediato
- alcuni test assumono che esistano modelli locali o il validator compilato

Conclusione pratica:
- questa repo e ottima come base per capire l'architettura del benchmark e per analizzare risultati gia prodotti
- per rieseguire tutto da zero serve reintegrare almeno i pesi locali dei modelli e il validator VAL


## 8. Perche questo notebook e qui

Se dovessi scegliere il posto giusto per un notebook come questo, lo metterei esattamente qui dove si trova ora:

```text
LLM-Needs-a-Plan-main/Repository Walkthrough.ipynb
```

Motivazione:
- e un documento trasversale a tutta la repo, non a un solo modulo
- e vicino a `README.md`, `report.pdf` e `Results Analysis.ipynb`
- chi apre il sottoprogetto vede subito sia il notebook di analisi risultati sia il notebook di orientamento alla struttura

Se in futuro la repo crescesse molto, un'alternativa sensata sarebbe una cartella `docs/` o `notebooks/`. Nella struttura attuale, la root del sottoprogetto resta la scelta piu pulita.


In [3]:
from pathlib import Path


def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / 'config.yml').exists() and (candidate / 'src').exists():
            return candidate
    raise FileNotFoundError('Could not locate the project root from the current working directory.')


PROJECT_ROOT = find_project_root()
PROJECT_ROOT


WindowsPath('C:/Users/utente/OneDrive/Documenti/GitHub/LLM_Benchmark/References/LLM-Needs-a-Plan-main/LLM-Needs-a-Plan-main')

In [2]:
import os
from pathlib import Path


def print_tree(root: Path, max_depth: int = 2) -> None:
    root = root.resolve()
    for current, dirs, files in os.walk(root):
        rel = Path(current).relative_to(root)
        depth = 0 if str(rel) == '.' else len(rel.parts)
        if depth > max_depth:
            dirs[:] = []
            continue
        indent = '  ' * depth
        label = root.name if depth == 0 else Path(current).name
        print(f'{indent}{label}/')
        if depth == max_depth:
            continue
        for file_name in sorted(files):
            print(f'{indent}  {file_name}')


print_tree(PROJECT_ROOT, max_depth=2)


LLM-Needs-a-Plan-main/
  .gitignore
  README.md
  Repository Walkthrough.ipynb
  Results Analysis.ipynb
  config.yml
  report.pdf
  requirements.txt
  requirements_analysis.txt
  assets/
    Guida Leonardo.pdf
    images/
    literature/
    tutorials/
  scripts/
    gemma3_iters_1.sh
    gemma3_iters_2.sh
    gemma3_iters_3.sh
    gemma3_iters_4.sh
    gemma3_iters_4_citycar.sh
    gemma3_iters_4_tetris.sh
    kimi_iters_1.sh
    kimi_iters_2.sh
    kimi_iters_3.sh
    kimi_iters_4.sh
    kimi_iters_4_citycar.sh
    kimi_iters_4_tetris.sh
    llama3_iters_1.sh
    llama3_iters_2.sh
    llama3_iters_3.sh
    llama3_iters_4.sh
    phi4_iters_1.sh
    phi4_iters_2.sh
    phi4_iters_3.sh
    phi4_iters_4.sh
  src/
    main.py
    core/
    data/
    prompts/
    results/
    tests/
    utils/


In [ ]:
from pathlib import Path


results_root = PROJECT_ROOT / 'src' / 'results'
rows = []

for plan_file in results_root.rglob('*_plan.txt'):
    content = plan_file.read_text(encoding='utf-8')
    parts = content.split('--- Processing Metadata ---')
    plan_text = parts[0]
    metadata = {}
    if len(parts) > 1:
        for line in parts[1].splitlines():
            if ':' in line:
                key, value = line.split(':', 1)
                metadata[key.strip()] = value.strip()

    actions = [
        line.strip()
        for line in plan_text.splitlines()
        if line.strip().startswith('(') and line.strip().endswith(')')
    ]

    rows.append(
        {
            'model': plan_file.parent.parent.name,
            'domain': plan_file.parent.name,
            'problem': plan_file.stem.replace('_plan', ''),
            'valid': metadata.get('Plan Valid', 'False').lower() == 'true',
            'iterations': metadata.get('Iterations', 'N/A'),
            'plan_length': len(actions),
        }
    )

summary = {}
for row in rows:
    key = (row['model'], row['domain'])
    summary.setdefault(key, {'files': 0, 'valid': 0})
    summary[key]['files'] += 1
    summary[key]['valid'] += int(row['valid'])

for (model, domain), data in sorted(summary.items()):
    rate = (data['valid'] / data['files'] * 100.0) if data['files'] else 0.0
    print(f"{model:7} | {domain:7} | {data['valid']:2}/{data['files']:2} | {rate:5.1f}%")
